# idealista18 and the .rda format

## Where the data comes from

[idealista](https://www.idealista.com) is one of the main property portals in Spain. In 2024, two authors from idealista and two academics published **idealista18**, a large sample of the portal's 2018 listings for the three largest Spanish cities, as an open data product:

> Rey-Blanco, D., Arbués, P., López, F. & Páez, A. (2024). A geo-referenced micro-data set of real estate listings for Spain's three largest cities. *Environment and Planning B: Urban Analytics and City Science*. https://doi.org/10.1177/23998083241242844

The data is distributed as an R package, in the repository [github.com/paezha/idealista18](https://github.com/paezha/idealista18). For each city it contains three objects:

| Object | Content |
| --- | --- |
| `<City>_Sale` | The listings for sale: one row per ad and quarter, 42 variables. Madrid has 94,815 rows, Barcelona 61,486 and Valencia 33,622 (189,923 in total) |
| `<City>_Polygons` | The boundaries of the city's neighbourhoods, as used by idealista |
| `<City>_POIS` | Points of interest: the city centre, the course of a main street, and the metro stations |

The listings are the data. The polygons and the points of interest are *reference data*: the ETL pipeline uses them to enrich the listings and to check them.

Four properties of the listings matter for everything that follows:

- **Asking prices, not sale prices.** `PRICE` is what the advertiser asked for on the portal, not what the buyer finally paid. A model trained on it predicts asking prices.
- **Quarterly snapshots.** The ads were extracted once per quarter. `PERIOD` encodes the quarter as `YYYYMM` with the quarter's last month: `201803`, `201806`, `201809`, `201812`.
- **Two sources.** Most columns come from the ad itself, filled in by the advertiser. The columns prefixed `CAD` come from the Spanish cadastre (*Catastro*), joined by the authors, and can disagree with the ad: `CADCONSTRUCTIONYEAR` is not always `CONSTRUCTIONYEAR`.
- **Anonymised on purpose.** For legal reasons the authors perturbed every listing before publishing it ([method and code](https://github.com/paezha/idealista18/tree/master/anonymization)): each price was multiplied by a random factor between 0.975 and 1.025 and rounded to the nearest €1,000, and each location was moved 30 to 60 m in a random direction, without leaving its neighbourhood. No model can recover that noise, so it sets a floor under the error any model can reach on this data.

The licence is the [Open Database License (ODbL) v1.0](https://opendatacommons.org/licenses/odbl/1-0/): use is free, with **attribution** (the citation above) and **share-alike**: a database derived from it and used in public must be offered under the same licence. Every table built from it in this repository travels with the citation and the licence.

## Downloading the source files

The nine files are downloaded from the repository's `master` branch and cached in `data/raw_rda/`, so a second run doesn't download them again. `master` is a moving reference: if the authors ever changed the data, the row counts checked in the next notebook would reveal it. The whole `data/` folder is kept out of version control: datasets are downloaded or rebuilt, never committed.

In [1]:
from pathlib import Path

DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw_rda"

CITIES = ("Madrid", "Barcelona", "Valencia")
OBJECT_KINDS = ("Sale", "Polygons", "POIS")
SOURCE_URL_TEMPLATE = "https://github.com/paezha/idealista18/raw/master/data/{file_name}"

In [2]:
import urllib.request

RAW_DIR.mkdir(parents=True, exist_ok=True)
for city in CITIES:
    for kind in OBJECT_KINDS:
        destination = RAW_DIR / f"{city}_{kind}.rda"
        if not destination.exists():
            urllib.request.urlretrieve(SOURCE_URL_TEMPLATE.format(file_name=destination.name), destination)
        print(f"{destination.name:<24} {destination.stat().st_size / 1e6:6.2f} MB")

Madrid_Sale.rda            6.74 MB
Madrid_Polygons.rda        0.07 MB
Madrid_POIS.rda            0.01 MB
Barcelona_Sale.rda         4.36 MB
Barcelona_Polygons.rda     0.04 MB
Barcelona_POIS.rda         0.01 MB
Valencia_Sale.rda          2.41 MB
Valencia_Polygons.rda      0.03 MB
Valencia_POIS.rda          0.00 MB


## What an .rda file is

R can save objects from its workspace to disk with `save()` and restore them with `load()`. The file format is called **RData**, with the extension `.RData` or `.rda`: a binary snapshot of one or more *named* R objects, which `load()` brings back under the same names. Its sibling `.rds` (written by `saveRDS()`, read by `readRDS()`) stores a single object without a name.

R packages ship their datasets as `.rda` files in a `data/` folder. That is why idealista18 comes in this format: an R user types `library(idealista18)` and `Madrid_Sale` is simply there. For anyone outside R it is an obstacle: none of pandas' readers understands it. Before converting anything, it is worth looking at the raw bytes.

In [3]:
rda_bytes = (RAW_DIR / "Valencia_Sale.rda").read_bytes()
rda_bytes[:10]

b'BZh91AY&SY'

`BZh` is the signature of a **bzip2**-compressed file, and `9` is its block size (900 KB, the maximum). The next six bytes, `1AY&SY`, mark the start of a compressed block: they are `0x314159265359`, the first digits of π. R compresses `.rda` files when it writes them; `save()` uses gzip by default, and packages often choose bzip2 or xz, which compress better. Python's standard library decompresses bzip2:

In [4]:
import bz2

decompressed_bytes = bz2.decompress(rda_bytes)
print(f"{len(rda_bytes) / 1e6:.1f} MB compressed, {len(decompressed_bytes) / 1e6:.1f} MB decompressed")
decompressed_bytes[:40]

2.4 MB compressed, 9.9 MB decompressed


b'RDX2\nX\n\x00\x00\x00\x02\x00\x04\x02\x02\x00\x02\x03\x00\x00\x00\x04\x02\x00\x00\x00\x01\x00\x04\x00\t\x00\x00\x00\rValen'

Under the compression is the RData format itself:

- `RDX2\n`: an RData file, version 2 of the serialization format.
- `X\n`: the encoding, here **XDR**, a portable binary standard with big-endian numbers. The alternatives are `A` (text) and `B` (the writing machine's native binary).
- Three 4-byte integers: the format version, the version of R that wrote the file, and the oldest version of R that can read it. R packs a version number `x.y.z` as `x·65536 + y·256 + z`.

In [5]:
import struct


def unpack_r_version(packed_version: int) -> str:
    return f"{packed_version >> 16}.{(packed_version >> 8) & 0xFF}.{packed_version & 0xFF}"


header_length = len(b"RDX2\nX\n")
format_version, writer_version, minimum_reader_version = struct.unpack(
    ">III", decompressed_bytes[header_length:header_length + 12])  # ">": big-endian, as XDR requires
print(f"format {format_version}, written by R {unpack_r_version(writer_version)}, "
      f"readable by R >= {unpack_r_version(minimum_reader_version)}")

format 2, written by R 4.2.2, readable by R >= 2.3.0


After the header comes the content: a list of named objects, each one stored as a tree of R values, every value preceded by a code for its type. The name of the first (and only) object, `Valencia_Sale`, is already visible in the 40 bytes printed above; the data frame itself follows it, column by column.

Because the format is documented, it can be read without installing R. There are three common routes:

- **R itself:** `load("Valencia_Sale.rda")` followed by `write.csv(...)`. Reliable, but it adds R to the toolchain.
- **[pyreadr](https://github.com/ofajardo/pyreadr):** wraps the C library librdata. Fast, and enough for plain data frames, but it does not cover every kind of R object.
- **[rdata](https://github.com/vnmabus/rdata):** a pure-Python parser for the whole format. The idealista18 repository's own Python example uses it, and so does this notebook.

## Reading an .rda file from Python

`rdata` works in two steps. **Parsing** turns the bytes into a generic tree of R objects, exactly as stored: types, attributes and R class names, with no interpretation yet. **Conversion** then maps each R class onto a Python equivalent: an R `data.frame` becomes a pandas DataFrame, a `factor` becomes a categorical column, and so on.

In [6]:
import rdata

parsed = rdata.parser.parse_file(RAW_DIR / "Valencia_Sale.rda")
parsed.versions

RVersions(format=2, serialized=262658, minimum=131840)

The parser reads the same header decoded by hand above: format 2, written by R 4.2.2 (`0x40202`), readable by R 2.3.0 or later (`0x20300`). Converting the tree produces many warnings, so the next cell records them instead of printing them, and shows the distinct messages.

In [7]:
import warnings

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    converted = rdata.conversion.convert(parsed)

print(f"{len(caught_warnings):,} warnings, {len({str(warning.message) for warning in caught_warnings})} distinct")
sorted({str(warning.message) for warning in caught_warnings})

100,871 warnings, 8 distinct


['Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.',
 'Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.',
 'Missing constructor for R class "bbox". The underlying R object is returned instead.',
 'Missing constructor for R class "crs". The underlying R object is returned instead.',
 'Missing constructor for R class "sf". The constructor for class "data.frame" will be used instead.',
 'Missing constructor for R class "sfc". The underlying R object is returned instead.',
 'Missing constructor for R class "sfc_POINT". The constructor for class "sfc" will be used instead.',
 'Missing constructor for R class "sfg". The underlying R object is returned instead.']

Every message is about a class from **sf** (*simple features*), R's standard package for spatial vector data. `Valencia_Sale` is not a plain data frame but an `sf` object: a data frame with an extra `geometry` column that holds one `POINT` per row, plus a coordinate reference system (`crs`) and a bounding box (`bbox`). `rdata` has no Python constructors for those classes, so for each of them it falls back to the nearest class it knows, or to the raw R object; it warns once per value, which is why the count is so large.

The fallback is harmless here, for a reason worth checking rather than assuming: the coordinates are also stored as two plain numeric columns, `LONGITUDE` and `LATITUDE`.

In [8]:
valencia_listings = converted["Valencia_Sale"]
print(type(valencia_listings).__name__, valencia_listings.shape)
valencia_listings[["LONGITUDE", "LATITUDE", "geometry"]].head(3)

DataFrame (33622, 42)


,LONGITUDE,LATITUDE,geometry
1,-0.393538,39.500908,"[-0.393538331726532, 39.5009075674168]"
2,-0.359125,39.482599,"[-0.359124975348099, 39.4825994457946]"
3,-0.362470,39.485144,"[-0.362469934782961, 39.4851438187497]"


In [9]:
import numpy as np

geometry_points = np.vstack(valencia_listings["geometry"].to_numpy())
np.array_equal(geometry_points, valencia_listings[["LONGITUDE", "LATITUDE"]].to_numpy())

True

Each geometry is exactly the (longitude, latitude) pair of its row, so the `geometry` column adds nothing and can be dropped. If it were the only source of the coordinates, decoding it would be a real step of the extraction.

### What the conversion produced

In [10]:
valencia_listings.dtypes.value_counts()

Int32       33
float64      7
category     1
object       1
Name: count, dtype: int64

The conversion keeps R's types faithfully, and three of them deserve a note:

- **`Int32`**, with a capital I, is pandas' *nullable* 32-bit integer. R integers are 32-bit and can be `NA`, while a NumPy `int64` column cannot hold a missing value; `Int32` can.
- **`category`** comes from an R `factor`: here `ASSETID`, the listing identifier, stored as a set of labels plus one code per row.
- **`object`** is the `geometry` column, one NumPy array per row.

The index comes from R as well: R numbers rows from 1.

In [11]:
valencia_listings.index[:5]

RangeIndex(start=1, stop=6, step=1)

## The reference objects

### Neighbourhood polygons

`rdata.read_rda` runs the parse and conversion steps in one call. The warnings are filtered by their message, not silenced wholesale: only the "missing constructor" warnings just examined are expected, and any other warning should still show up. The polygon files hold neighbourhood names with accents, and a first attempt to read one fails:

In [12]:
def read_without_sf_warnings(path: Path, **read_options) -> dict:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Missing constructor for R class")
        return rdata.read_rda(path, **read_options)


try:
    with warnings.catch_warnings(record=True) as decoding_warnings:
        warnings.simplefilter("always")
        read_without_sf_warnings(RAW_DIR / "Valencia_Polygons.rda")
except UnicodeDecodeError as error:
    print(f"{type(error).__name__}: {error}")
print(decoding_warnings[0].message)

UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 in position 5: ordinal not in range(128)
Unknown encoding. Assumed ASCII.


R marks every string with its encoding (UTF-8, Latin-1, ASCII), or leaves it unmarked, meaning "the native encoding of the machine that wrote it". These names are unmarked, and `rdata` decodes unmarked strings as ASCII unless told otherwise, as its first warning says, so the first accented letter (`0xc3`, the first byte of a UTF-8 `í` or `à`) breaks it. The bytes are UTF-8, so the fix is to say so:

In [13]:
valencia_polygons = read_without_sf_warnings(RAW_DIR / "Valencia_Polygons.rda",
                                             default_encoding="utf8")["Valencia_Polygons"]
valencia_polygons.drop(columns="geometry").head()

,LOCATIONID,LOCATIONNAME,ZONELEVELID
1,0-EU-ES-46-02-002-250-01-001,La Seu,8
2,0-EU-ES-46-02-002-250-01-006,Sant Francesc,8
3,0-EU-ES-46-02-002-250-06-003,Jaume Roig,8
4,0-EU-ES-46-02-002-250-03-002,La Roqueta,8
5,0-EU-ES-46-02-002-250-04-002,Les Tendetes,8


Each row is a neighbourhood: idealista's identifier, its name, and a zone level (8 for every row, so it carries no information here). The geometry is again an `sf` value converted to plain Python: a **multipolygon**, that is, a list of polygons, each a list of rings, each ring an array of (longitude, latitude) vertices. The first ring of a polygon is its outer boundary; any further rings are holes.

In [14]:
first_neighbourhood = valencia_polygons["geometry"].iloc[0]
print(f"{len(first_neighbourhood)} polygon(s); rings per polygon: {[len(polygon) for polygon in first_neighbourhood]}")
first_neighbourhood[0][0][:3]

1 polygon(s); rings per polygon: [1]


array([[-0.373611, 39.480499],
       [-0.374036, 39.479979],
       [-0.374526, 39.479376]])

### Points of interest

A points-of-interest object is not a data frame but an R list, which `rdata` converts into a dictionary of small data frames:

In [15]:
valencia_pois = read_without_sf_warnings(RAW_DIR / "Valencia_POIS.rda")["Valencia_POIS"]
{str(name): table.shape for name, table in valencia_pois.items()}

{'City_Center': (1, 2), 'Metro': (99, 2), 'Blasco': (27, 2)}

`City_Center` is a single point, `Metro` the metro stations, and `Blasco` a sequence of points that traces Avenida de Blasco Ibáñez, Valencia's reference street. Madrid's list has `Castellana` and Barcelona's `Diagonal` instead. The listings' columns `DISTANCE_TO_CITY_CENTER`, `DISTANCE_TO_METRO` and `DISTANCE_TO_BLASCO` were computed from these points, which the next notebook checks.

In [16]:
valencia_pois["City_Center"]

,Lon,Lat
1,-0.376619,39.469766
